# Задача 2: Нейронные сети и анализ тональности

Реализация нейронной сети с нуля на numpy. Без pytorch/tensorflow.
Задача — классифицировать текстовые отзывы по тональности.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## Данные

Используем небольшой датасет отзывов на русском языке.
Три класса: 0 — отрицательный, 1 — положительный, 2 — нейтральный.

In [ ]:
sentiment_data = {
    'text': [
        # положительные
        'Отличный фильм, очень понравился!',
        'Прекрасная книга, рекомендую всем прочитать.',
        'Качественный товар, доволен покупкой.',
        'Замечательный сервис, быстро и профессионально.',
        'Потрясающий ресторан, вкусная еда и уютная атмосфера.',
        'Великолепный спектакль, получил массу удовольствия.',
        'Отличное приложение, удобный интерфейс.',
        'Прекрасный отель, комфортные номера и хороший завтрак.',
        'Замечательный преподаватель, интересные лекции.',
        'Потрясающая музыка, слушаю постоянно.',
        'Отличный телефон, все работает быстро.',
        'Прекрасный подарок, очень доволен.',
        'Качественный кофе, ароматный и вкусный.',
        'Замечательный парк, красивая природа.',
        'Потрясающий концерт, незабываемые эмоции.',
        'Супер игра, затягивает с первых минут.',
        'Восхитительный вид, красота природы просто поражает.',
        'Лучший ноутбук из всех что у меня был.',
        'Очень рад покупке, рекомендую всем.',
        'Великолепное качество за разумную цену.',

        # отрицательные
        'Ужасный фильм, полная потеря времени.',
        'Плохая книга, скучно и неинтересно.',
        'Низкое качество товара, очень разочарован.',
        'Плохой сервис, долго ждал и ничего не получил.',
        'Отвратительный ресторан, невкусная еда.',
        'Скучный спектакль, не рекомендую.',
        'Плохое приложение, постоянно глючит.',
        'Ужасный отель, грязные номера и плохое обслуживание.',
        'Скучный преподаватель, неинтересные лекции.',
        'Плохая музыка, не понравилась.',
        'Плохой телефон, постоянно зависает.',
        'Разочаровался в подарке, не то что ожидал.',
        'Плохой кофе, безвкусный и холодный.',
        'Запущенный парк, грязно и неухоженно.',
        'Скучный концерт, не оправдал ожиданий.',
        'Ужасная игра, трата денег.',
        'Разочарован сервисом, больше не вернусь.',
        'Плохое качество, сломалось через неделю.',
        'Отстойный фильм, жалею о потраченном времени.',
        'Кошмарный опыт, никогда больше туда не пойду.',

        # нейтральные
        'Обычный фильм, ничего особенного.',
        'Стандартная книга, читается нормально.',
        'Обычный товар, соответствует описанию.',
        'Нормальный сервис, без особых претензий.',
        'Обычный ресторан, стандартное меню.',
        'Средний спектакль, можно посмотреть.',
        'Обычное приложение, работает как надо.',
        'Стандартный отель, без изысков.',
        'Обычный преподаватель, стандартные лекции.',
        'Средняя музыка, ничего особенного.',
        'Нормальный телефон, делает своё дело.',
        'Ожидаемое качество за эту цену.',
        'Средний концерт, нормально провел время.',
        'Стандартная покупка, всё соответствует описанию.',
        'Обычная поездка, без приключений.',
        'Среднее качество, не лучше и не хуже.',
        'Нейтральное впечатление, ни хорошо ни плохо.',
        'Среднестатистический продукт для своей цены.',
        'Нормально, но ничего выдающегося.',
        'Обычный опыт, как и ожидалось.',
    ],
    'sentiment': [
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
    ]
}

df = pd.DataFrame(sentiment_data)
print(f'Всего отзывов: {len(df)}')
print(df['sentiment'].value_counts().sort_index())
print(df.head(5))

## Векторизация текстов

TF-IDF переводит тексты в числовые векторы. Берём уни- и биграммы, 200 признаков.
После этого нормализуем через StandardScaler — нейронки плохо работают с разными масштабами признаков.

In [ ]:
vectorizer = TfidfVectorizer(max_features=200, ngram_range=(1, 2), sublinear_tf=True)
X = vectorizer.fit_transform(df['text']).toarray()
y_multi = df['sentiment'].values

print(f'Размер матрицы признаков: {X.shape}')

# сплит для мультиклассовой задачи (все 3 класса)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X, y_multi, test_size=0.25, random_state=42, stratify=y_multi
)
scaler = StandardScaler()
X_train_m = scaler.fit_transform(X_train_m)
X_test_m = scaler.transform(X_test_m)

# сплит для бинарной задачи (убираем нейтральные)
mask = y_multi != 2
X_bin, y_bin = X[mask], y_multi[mask]
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bin, y_bin, test_size=0.25, random_state=42, stratify=y_bin
)
scaler_b = StandardScaler()
X_train_b = scaler_b.fit_transform(X_train_b)
X_test_b = scaler_b.transform(X_test_b)

print(f'Мультикласс — train: {X_train_m.shape}, test: {X_test_m.shape}')
print(f'Бинарная    — train: {X_train_b.shape}, test: {X_test_b.shape}')

## Реализация нейронной сети

Всё написано вручную на numpy. Реализовано:
- forward pass через произвольное число слоёв
- backpropagation с цепным правилом
- функции потерь: binary cross-entropy и categorical cross-entropy
- dropout (inverted)
- early stopping

In [ ]:
class NeuralNetwork:

    def __init__(self, layers, activation='relu', dropout_rate=0.0,
                 task_type='binary', num_classes=2):
        # layers — список с количеством нейронов в каждом слое
        # последний элемент = выходной слой
        self.layers = layers
        self.activation = activation
        self.dropout_rate = dropout_rate
        self.task_type = task_type
        self.num_classes = num_classes

        self.weights = []
        self.biases = []
        self.train_losses = []
        self.val_losses = []
        self._training = False

    def _init_weights(self, input_dim):
        # инициализация весов по схеме He (для relu) или Xavier (для sigmoid)
        # плохая инициализация — сеть вообще не учится или очень медленно
        dims = [input_dim] + self.layers
        self.weights = []
        self.biases = []
        for i in range(len(dims) - 1):
            scale = np.sqrt(2.0 / dims[i]) if self.activation == 'relu' else np.sqrt(1.0 / dims[i])
            self.weights.append(np.random.randn(dims[i], dims[i + 1]) * scale)
            self.biases.append(np.zeros((1, dims[i + 1])))

    # функции активации
    @staticmethod
    def _relu(z):
        return np.maximum(0, z)

    @staticmethod
    def _relu_deriv(z):
        # производная relu: 1 там где z > 0, иначе 0
        return (z > 0).astype(float)

    @staticmethod
    def _sigmoid(z):
        z = np.clip(z, -500, 500)  # clip чтобы не было overflow
        return 1.0 / (1.0 + np.exp(-z))

    @staticmethod
    def _sigmoid_deriv(a):
        return a * (1.0 - a)

    @staticmethod
    def _softmax(z):
        # вычитаем max для численной стабильности
        e = np.exp(z - np.max(z, axis=1, keepdims=True))
        return e / e.sum(axis=1, keepdims=True)

    def _activate(self, z):
        return self._relu(z) if self.activation == 'relu' else self._sigmoid(z)

    def _activate_deriv(self, a):
        return self._relu_deriv(a) if self.activation == 'relu' else self._sigmoid_deriv(a)

    def _forward_pass(self, X):
        # прямой проход: для каждого слоя считаем z = A_prev @ W + b, потом применяем активацию
        # для выходного слоя — sigmoid (binary) или softmax (multiclass)
        # во время обучения применяем dropout к скрытым слоям
        zs = []
        activations = [X]
        masks = []

        n_layers = len(self.weights)
        for i, (W, b) in enumerate(zip(self.weights, self.biases)):
            z = activations[-1] @ W + b
            zs.append(z)

            if i == n_layers - 1:
                # выходной слой
                a = self._sigmoid(z) if self.task_type == 'binary' else self._softmax(z)
                masks.append(None)
            else:
                # скрытый слой
                a = self._activate(z)
                # dropout: случайно обнуляем нейроны и масштабируем оставшиеся
                if self._training and self.dropout_rate > 0:
                    mask = (np.random.rand(*a.shape) > self.dropout_rate).astype(float)
                    a = a * mask / (1.0 - self.dropout_rate)  # inverted dropout
                    masks.append(mask)
                else:
                    masks.append(None)

            activations.append(a)

        return zs, activations, masks

    def _compute_loss(self, y_true, y_pred):
        # logloss для бинарной: -mean(y*log(p) + (1-y)*log(1-p))
        # categorical cross-entropy для мультиклассовой
        eps = 1e-15
        if self.task_type == 'binary':
            p = np.clip(y_pred, eps, 1 - eps)
            return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))
        else:
            p = np.clip(y_pred, eps, 1.0)
            return -np.mean(np.sum(y_true * np.log(p), axis=1))

    def _backward_pass(self, X, y, zs, activations, masks):
        # backpropagation — считаем градиенты через цепное правило
        # начинаем с выходного слоя и идём назад
        # для softmax+CCE и sigmoid+BCE градиент на выходе одинаков: y_pred - y_true
        n = X.shape[0]
        n_layers = len(self.weights)
        grads_W = [None] * n_layers
        grads_b = [None] * n_layers

        delta = activations[-1] - y  # градиент ошибки на выходном слое

        for i in reversed(range(n_layers)):
            A_prev = activations[i]
            grads_W[i] = A_prev.T @ delta / n
            grads_b[i] = delta.mean(axis=0, keepdims=True)

            if i > 0:
                # передаём градиент на предыдущий слой
                delta = delta @ self.weights[i].T
                # учитываем dropout-маску если была
                if masks[i - 1] is not None:
                    delta = delta * masks[i - 1] / (1.0 - self.dropout_rate)
                # умножаем на производную функции активации
                delta = delta * self._activate_deriv(activations[i])

        return grads_W, grads_b

    def fit(self, X, y, epochs=200, batch_size=32, learning_rate=0.01,
            validation_data=None, early_stopping=False, patience=15, verbose=True):

        self._init_weights(X.shape[1])
        self.train_losses = []
        self.val_losses = []

        # для мультиклассовой переводим метки в one-hot
        y_fit = self._to_onehot(y) if self.task_type == 'multiclass' else y.reshape(-1, 1).astype(float)

        best_val_loss = np.inf
        patience_counter = 0
        best_weights = None
        best_biases = None

        for epoch in range(epochs):
            self._training = True

            # перемешиваем данные перед каждой эпохой
            idx = np.random.permutation(X.shape[0])
            X_shuf, y_shuf = X[idx], y_fit[idx]

            epoch_losses = []
            for start in range(0, X.shape[0], batch_size):
                Xb = X_shuf[start:start + batch_size]
                yb = y_shuf[start:start + batch_size]

                zs, activations, masks = self._forward_pass(Xb)
                batch_loss = self._compute_loss(yb, activations[-1])
                epoch_losses.append(batch_loss)

                grads_W, grads_b = self._backward_pass(Xb, yb, zs, activations, masks)

                # обновляем веса — градиентный спуск
                for i in range(len(self.weights)):
                    self.weights[i] -= learning_rate * grads_W[i]
                    self.biases[i] -= learning_rate * grads_b[i]

            train_loss = np.mean(epoch_losses)
            self.train_losses.append(train_loss)
            self._training = False

            # считаем val_loss и проверяем early stopping
            if validation_data is not None:
                X_val, y_val = validation_data
                y_val_fit = self._to_onehot(y_val) if self.task_type == 'multiclass' \
                    else y_val.reshape(-1, 1).astype(float)
                _, val_activations, _ = self._forward_pass(X_val)
                val_loss = self._compute_loss(y_val_fit, val_activations[-1])
                self.val_losses.append(val_loss)

                if early_stopping:
                    if val_loss < best_val_loss - 1e-6:
                        best_val_loss = val_loss
                        patience_counter = 0
                        # сохраняем лучшие веса
                        best_weights = [w.copy() for w in self.weights]
                        best_biases = [b.copy() for b in self.biases]
                    else:
                        patience_counter += 1
                        if patience_counter >= patience:
                            if verbose:
                                print(f'Early stopping на эпохе {epoch + 1}, лучший val_loss={best_val_loss:.4f}')
                            # возвращаем лучшие веса
                            self.weights = best_weights
                            self.biases = best_biases
                            break

            if verbose and (epoch + 1) % 20 == 0:
                msg = f'Эпоха {epoch + 1}/{epochs}  train_loss={train_loss:.4f}'
                if validation_data is not None:
                    msg += f'  val_loss={val_loss:.4f}'
                print(msg)

        return self

    def predict_proba(self, X):
        # forward pass без dropout, возвращаем вероятности
        self._training = False
        _, activations, _ = self._forward_pass(X)
        return activations[-1]

    def predict(self, X):
        proba = self.predict_proba(X)
        if self.task_type == 'binary':
            return (proba.flatten() >= 0.5).astype(int)
        else:
            return np.argmax(proba, axis=1)

    def _to_onehot(self, y):
        oh = np.zeros((y.shape[0], self.num_classes))
        oh[np.arange(y.shape[0]), y.astype(int)] = 1.0
        return oh


print('Класс NeuralNetwork готов')

## Бинарная классификация (положительный / отрицательный)

In [ ]:
nn_binary = NeuralNetwork(
    layers=[64, 32, 1],
    activation='relu',
    dropout_rate=0.3,
    task_type='binary'
)

nn_binary.fit(
    X_train_b, y_train_b,
    epochs=300,
    batch_size=16,
    learning_rate=0.005,
    validation_data=(X_test_b, y_test_b),
    early_stopping=True,
    patience=20
)

y_pred_b = nn_binary.predict(X_test_b)

print(classification_report(y_test_b, y_pred_b, target_names=['Отрицательный', 'Положительный']))

f1_b = f1_score(y_test_b, y_pred_b)
p_b = precision_score(y_test_b, y_pred_b)
r_b = recall_score(y_test_b, y_pred_b)
print(f'F1-score : {f1_b:.4f}')
print(f'Precision: {p_b:.4f}')
print(f'Recall   : {r_b:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# график лосса
axes[0].plot(nn_binary.train_losses, label='train')
axes[0].plot(nn_binary.val_losses, label='val', linestyle='--')
axes[0].set_title('Бинарная: loss по эпохам')
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Binary Cross-Entropy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# матрица ошибок
cm = confusion_matrix(y_test_b, y_pred_b)
axes[1].imshow(cm, cmap='Blues')
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(['Отриц.', 'Полож.'])
axes[1].set_yticklabels(['Отриц.', 'Полож.'])
axes[1].set_xlabel('Предсказано')
axes[1].set_ylabel('Истинно')
axes[1].set_title('Матрица ошибок')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, cm[i, j], ha='center', va='center', fontsize=16)

plt.tight_layout()
plt.show()

## Мультиклассовая классификация (3 класса)

In [ ]:
nn_multi = NeuralNetwork(
    layers=[128, 64, 32, 3],
    activation='relu',
    dropout_rate=0.3,
    task_type='multiclass',
    num_classes=3
)

nn_multi.fit(
    X_train_m, y_train_m,
    epochs=400,
    batch_size=16,
    learning_rate=0.005,
    validation_data=(X_test_m, y_test_m),
    early_stopping=True,
    patience=25
)

y_pred_m = nn_multi.predict(X_test_m)

print(classification_report(y_test_m, y_pred_m,
      target_names=['Отрицательный', 'Положительный', 'Нейтральный']))

f1_m = f1_score(y_test_m, y_pred_m, average='macro')
p_m = precision_score(y_test_m, y_pred_m, average='macro')
r_m = recall_score(y_test_m, y_pred_m, average='macro')
print(f'F1-score  (macro): {f1_m:.4f}')
print(f'Precision (macro): {p_m:.4f}')
print(f'Recall    (macro): {r_m:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(nn_multi.train_losses, label='train')
axes[0].plot(nn_multi.val_losses, label='val', linestyle='--')
axes[0].set_title('Мультикласс: loss по эпохам')
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Categorical Cross-Entropy')
axes[0].legend()
axes[0].grid(alpha=0.3)

cm_m = confusion_matrix(y_test_m, y_pred_m)
axes[1].imshow(cm_m, cmap='Greens')
labels = ['Отриц.', 'Полож.', 'Нейтр.']
axes[1].set_xticks([0, 1, 2])
axes[1].set_yticks([0, 1, 2])
axes[1].set_xticklabels(labels)
axes[1].set_yticklabels(labels)
axes[1].set_xlabel('Предсказано')
axes[1].set_ylabel('Истинно')
axes[1].set_title('Матрица ошибок')
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, cm_m[i, j], ha='center', va='center', fontsize=16)

plt.tight_layout()
plt.show()

## Сравнение с dropout и без

In [ ]:
results = []
for dr in [0.0, 0.3]:
    nn = NeuralNetwork(layers=[64, 32, 1], activation='relu',
                       dropout_rate=dr, task_type='binary')
    nn.fit(X_train_b, y_train_b, epochs=200, batch_size=16,
           learning_rate=0.005, verbose=False)
    preds = nn.predict(X_test_b)
    f1 = f1_score(y_test_b, preds)
    results.append((f'dropout={dr}', nn.train_losses, f1))
    print(f'dropout={dr}  F1={f1:.4f}')

plt.figure(figsize=(9, 4))
for label, losses, f1 in results:
    plt.plot(losses, label=f'{label} (F1={f1:.3f})')
plt.title('Train loss: с dropout vs без')
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Проверка на новых примерах

In [ ]:
new_reviews = [
    'Великолепно, буду рекомендовать всем!',
    'Полный провал, ужасное качество.',
    'Нормально, примерно как и ожидал.',
    'Лучший опыт в моей жизни, спасибо!',
    'Очень разочарован, больше не вернусь.',
]

X_new = vectorizer.transform(new_reviews).toarray()
X_new_m = scaler.transform(X_new)
X_new_b = scaler_b.transform(X_new)

preds_b = nn_binary.predict(X_new_b)
preds_m = nn_multi.predict(X_new_m)

bin_labels = {0: 'Отрицательный', 1: 'Положительный'}
multi_labels = {0: 'Отрицательный', 1: 'Положительный', 2: 'Нейтральный'}

print(f'{"Текст":<45} {"Бинарно":>15} {"Мультикласс":>15}')
print('-' * 77)
for text, pb, pm in zip(new_reviews, preds_b, preds_m):
    print(f'{text[:43]:<45} {bin_labels[pb]:>15} {multi_labels[pm]:>15}')

## Итоги

In [ ]:
print('Бинарная классификация:')
print(f'  F1={f1_b:.4f}  Precision={p_b:.4f}  Recall={r_b:.4f}')

print('\nМультиклассовая (macro):')
print(f'  F1={f1_m:.4f}  Precision={p_m:.4f}  Recall={r_m:.4f}')